# 11 Core Put Overlay Train (пут-оверлей для базовых стратегий на обучении)

Этот notebook (ноутбук) проверяет однослойный `put overlay (пут-оверлей)` для `self_sufficient_core (самодостаточных базовых стратегий)` на `train sample (обучающей выборке)`.

Запуск идет пакетами по 100 `core strategies (базовых стратегий)`, чтобы не убивать `kernel (ядро)` и память браузера.

## Grid (сетка параметров)

- `T (экспирация)`: 44, 220, 968 баров;
- `std span (окно волатильности)`: 44, 88;
- `K shift (сдвиг страйка)`: 0.5, 1.0, 1.5;
- `center mode (режим центра)`: `period_start (старт периода)`, `median_mad (медиана и медианное абсолютное отклонение)`;
- `overlay weight (вес оверлея)`: 0.25, 0.5, 1.0, 1.5.

Итого 144 варианта на одну `core strategy (базовую стратегию)`. Для 428 стратегий нужны `BATCH_ID (номера пакетов)` 0, 1, 2, 3, 4.

In [1]:
from pathlib import Path
import sys
import pandas as pd

WORK_ROOT = Path(r'./workspace')
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

from core_put_overlay_train import load_core_candidates, run_train_overlay_batch, aggregate_batches

candidates = load_core_candidates()
BATCH_SIZE = 100
N_BATCHES = (len(candidates) + BATCH_SIZE - 1) // BATCH_SIZE
len(candidates), BATCH_SIZE, N_BATCHES


(428, 100, 5)

## Run One Batch (запуск одного пакета)

Меняй только `BATCH_ID (номер пакета)`. Один запуск обрабатывает 100 стратегий и завершается.

In [3]:
for i in range(1, 5):
    BATCH_ID = i
    
    manifest = run_train_overlay_batch(batch_id=BATCH_ID, batch_size=BATCH_SIZE)
    manifest


## Aggregate Batches (агрегация пакетов)

Запускай после того, как обработаны все нужные `BATCH_ID (номера пакетов)`.

In [4]:
manifest = aggregate_batches()
manifest


{'n_candidates': 428,
 'n_grid_rows': 61632,
 'n_best_rows': 428,
 'n_errors': 0,
 'n_missing_spec_ids': 0,
 'output_dir': 'D:\\Filter_2_option_v2\\11_core_put_overlay_train_output',
 'saved_files': ['01_put_overlay_train_grid.csv',
  '02_put_overlay_train_best.csv',
  '03_put_overlay_train_best_paths.csv',
  '04_put_overlay_train_errors.csv',
  '05_put_overlay_train_instrument_summary.csv',
  '06_manifest.json',
  '07_missing_spec_ids.csv']}

## Best Results (лучшие результаты)

In [5]:
OUTPUT_DIR = Path(r'D:\Filter_2_option_v2\11_core_put_overlay_train_output')
best = pd.read_csv(OUTPUT_DIR / '02_put_overlay_train_best.csv')
best.head(30)


,spec_id,strategy_id,family,instrument,t_bars,std_span,k_shift_sigma,center_scale,overlay_weight,train_n_obs,...,overlay_train_sharpe,overlay_train_sortino,overlay_train_max_drawdown,overlay_improvement,drawdown_reduction,core_drawdown_to_pnl,overlay_drawdown_to_pnl,pnl_retention,overlay_score,source_file
0,201,RB__ma_cross__fast_44__slow_176,moving_average_crossover,RB,220,44,0.5,period_start,1.0,92610,...,0.338185,0.387384,-0.530069,0.068340,0.885755,1.822589,0.627182,1.087974,2.212395,D:\Filter_2_option_v2\11_core_put_overlay_trai...
1,812,E6_B6__pair_spread_mean_reversion__window_352_...,pair_spread_mean_reversion,E6/B6,968,44,1.0,period_start,1.5,96593,...,0.308229,0.277341,-1.547682,0.606941,0.883543,2.556149,0.993333,1.638127,2.748993,D:\Filter_2_option_v2\11_core_put_overlay_trai...
2,84,NG__time_series_momentum__lb_352,time_series_momentum,NG,220,44,0.5,median_mad,1.5,97984,...,0.263355,0.306354,-0.692774,0.506899,0.851285,3.532184,0.733841,2.159580,2.863992,D:\Filter_2_option_v2\11_core_put_overlay_trai...
3,508,A6_N6__pair_spread_mean_reversion__window_352_...,pair_spread_mean_reversion,A6/N6,968,88,1.5,period_start,1.0,93602,...,1.703190,1.207683,-0.643355,-1.321373,0.840379,0.205234,0.108894,0.817224,1.048849,D:\Filter_2_option_v2\11_core_put_overlay_trai...
4,798,E6_B6__pair_spread_mean_reversion__window_88__...,pair_spread_mean_reversion,E6/B6,44,44,1.5,period_start,1.5,96593,...,0.366802,0.394958,-1.740688,-0.262431,0.805167,1.015695,0.775679,0.895300,1.543064,D:\Filter_2_option_v2\11_core_put_overlay_trai...
5,691,D6_E6__pair_spread_momentum__window_88__z_0.5,pair_spread_momentum,D6/E6,968,44,0.5,period_start,1.5,96413,...,0.265280,0.304059,-0.557728,0.460207,0.731219,52.225915,1.150224,19.646783,9.691874,D:\Filter_2_option_v2\11_core_put_overlay_trai...
6,233,ZC__time_series_reversal__lb_88,time_series_reversal,ZC,968,44,0.5,period_start,1.5,74865,...,0.194052,0.222550,-0.577033,0.326834,0.712410,40.075296,1.607289,11.157861,6.085733,D:\Filter_2_option_v2\11_core_put_overlay_trai...
7,848,E6_N6__pair_spread_mean_reversion__window_352_...,pair_spread_mean_reversion,E6/N6,968,44,0.5,period_start,1.0,93507,...,0.310045,0.267863,-0.288759,0.000123,0.676982,2.236549,0.668544,1.000284,1.684533,D:\Filter_2_option_v2\11_core_put_overlay_trai...
8,386,SB__time_series_momentum__lb_176,time_series_momentum,SB,968,88,0.5,median_mad,1.5,44930,...,0.806145,0.917932,-0.567844,0.723381,0.664426,1.564831,0.375842,1.918605,2.821320,D:\Filter_2_option_v2\11_core_put_overlay_trai...
9,1164,S6_N6__pair_spread_mean_reversion__window_176_...,pair_spread_mean_reversion,S6/N6,220,44,1.5,period_start,1.0,93574,...,1.211339,1.492369,-0.715032,-0.477424,0.663463,0.237897,0.134478,0.917608,1.281586,D:\Filter_2_option_v2\11_core_put_overlay_trai...


## Missing Strategies (пропущенные стратегии)

In [6]:
pd.read_csv(OUTPUT_DIR / '07_missing_spec_ids.csv')


,spec_id


## Instruments (активы)

In [7]:
pd.read_csv(OUTPUT_DIR / '05_put_overlay_train_instrument_summary.csv').head(50)


,instrument,n,n_reduced_drawdown,n_improved_pnl,median_drawdown_reduction,median_overlay_improvement,median_overlay_total
0,ZL/ZW,9,9,2,7.871758e-02,-0.020292,1.020851
1,ZS/KE,9,9,4,6.555910e-02,-0.007596,0.245459
2,S6/N6,9,9,1,5.868714e-02,-0.113917,5.317090
3,D6/B6,9,9,1,4.898329e-02,-0.089412,1.334694
4,ZL/KE,9,9,2,4.405569e-02,-0.035735,1.358215
5,ZC/ZW,9,9,0,3.428173e-02,-0.033042,2.294967
6,HO/ZL,9,9,1,1.866206e-02,-0.018660,0.805854
7,RS/ZS,9,9,1,1.569966e-02,-0.010775,2.179461
8,LE/GF,9,9,2,1.295664e-02,-0.023681,1.300505
9,ZL/RS,9,9,0,2.457361e-03,-0.010278,1.741049
